<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Plan production checks

**[C05-M06-L03 · Plan production checks](https://learning.nextia-ai.com/courses/ml/m06/plan-production-checks/)** · C05, Machine Learning: From Problem to Reliable Model · Module 6, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** add input checks to `predict.py`, compute monitoring signals for new tickets, write a retraining trigger, and deliver the model with reproducible commands.

| | |
|---|---|
| **Time** | About 70 minutes |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | The artifact and `predict.py --check`, from [Package artifacts](https://learning.nextia-ai.com/courses/ml/m06/package-artifacts/). The model card, from [Document use boundaries](https://learning.nextia-ai.com/courses/ml/m06/document-use-boundaries/). |
| **You do not need** | The local project. The setup below downloads the data and the course's `ticket_model.py`, `train.py` and `evaluate.py`. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

In this module the **local project is the main route**. This notebook runs the same files as commands, in new processes.

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m06/plan-production-checks/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M06-L03-plan-production-checks/plan-production-checks-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the C05 practice data (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m01/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

The cell also downloads the course's finished project files `ticket_model.py`, `train.py`, `evaluate.py`, `requirements.txt` from [C05/project](https://github.com/usmanahmed99/nextia-learning-labs/tree/main/C05/project) into `ticket-model/`. The next cells train the model with `train.py`; the module practice also runs `evaluate.py` and copies `requirements.txt`.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = ["ticket_model.py", "train.py", "evaluate.py", "requirements.txt"]
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

## Setup: train the model and a helper to run commands

The next cell runs `train.py`, which writes `model/`. It also defines `run()`, which runs a project script in a new process and prints its output and its exit code (0 means success, 1 means the script stopped). You should see `Threshold 0.19`.

In [ ]:
import os, subprocess, sys

def run(*args, last=None, cwd=None):
    """Run a project script in a new process, like a command in a terminal,
    and print its output (or its last lines) and its exit code."""
    result = subprocess.run([sys.executable, *args], capture_output=True, text=True, cwd=cwd,
                            env={**os.environ, "PYTHON_COLORS": "0"})
    lines = (result.stdout + result.stderr).strip().splitlines()
    print("\n".join(lines[-last:] if last else lines))
    print(f"(exit code {result.returncode})")

run("train.py")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Three tickets, five inputs

The next cell writes the **first version** of `predict.py`, from Lesson 1, under another name: `predict_first.py`. It has no input checks. You compare it with the finished version.

In [ ]:
%%writefile predict_first.py
"""Score new tickets with the saved escalation model.

Nextia Learning, C05, Module 6 (Package artifacts; Plan production checks).

    python predict.py data/july_tickets.csv predictions.csv
    python predict.py --check     reload the model and compare the fixed sample
"""

import argparse
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from ticket_model import FEATURES, read

MODEL = Path("model")


def load_model():
    info = json.loads((MODEL / "model_info.json").read_text(encoding="utf-8"))
    return joblib.load(MODEL / "escalation_model.joblib"), info


def check_sample(pipeline):
    sample = read(MODEL / "check_sample.csv")
    scores = pipeline.predict_proba(sample[FEATURES])[:, 1]
    largest = float(np.abs(scores - sample["score"]).max())
    if largest > 1e-9:
        sys.exit(f"Check failed: the reloaded model's scores differ from the saved ones by up to {largest:.2e}.")
    print(f"Check passed: {len(sample)} scores are identical after reloading (largest difference {largest:.1e}).")


def main():
    parser = argparse.ArgumentParser(description="Score new tickets with the saved escalation model.")
    parser.add_argument("input", nargs="?", help="CSV file with one row per new ticket")
    parser.add_argument("output", nargs="?", default="predictions.csv")
    parser.add_argument("--check", action="store_true", help="compare the fixed sample after reloading")
    args = parser.parse_args()
    pipeline, info = load_model()
    if args.check:
        check_sample(pipeline)
        return
    if not args.input:
        parser.error("give an input CSV file, or --check")

    df = read(args.input)
    scores = pipeline.predict_proba(df[FEATURES])[:, 1]
    out = pd.DataFrame({"ticket_id": df["ticket_id"], "score": scores.round(4),
                        "flag": (scores >= info["threshold"]).astype(int)})
    out.to_csv(args.output, index=False)
    print(f"Scored {len(out)} tickets with threshold {info['threshold']}: {out['flag'].sum()} flagged "
          f"({out['flag'].mean():.1%}). Wrote {args.output}.")


if __name__ == "__main__":
    main()

The next cell takes the first three July tickets and makes five input files: the tickets as exported, and four broken copies.

In [ ]:
from ticket_model import FEATURES, TARGET, read

july = read("data/july_tickets.csv")
tiny = july.head(3)
tiny.to_csv("tiny_ok.csv", index=False)
social = tiny.copy(); social.loc[social.index[2], "channel"] = "social"; social.to_csv("tiny_social.csv", index=False)
tiny.drop(columns="prior_escalations_90d").to_csv("tiny_missing.csv", index=False)
text = tiny.astype({"priority": "string"}); text.loc[text.index[1], "priority"] = "high"; text.to_csv("tiny_text.csv", index=False)
cents = tiny.copy(); cents["order_value"] = cents["order_value"] * 100; cents.to_csv("tiny_cents.csv", index=False)
print(tiny[["ticket_id", "channel", "team", "priority", "order_value", "prior_escalations_90d"]])

> **Predict.** The first version has no checks. For each of the five files, will it score, fail with an error, or score with no message? Write your guesses down. Then run the cell. (It prints the last two lines of each run.)

In [ ]:
for name in ("tiny_ok", "tiny_social", "tiny_missing", "tiny_text", "tiny_cents"):
    print(f"--- {name}.csv")
    run("predict_first.py", f"{name}.csv", "tiny_out.csv", last=2)

> **Check.** `tiny_ok` and `tiny_social` both say `1 flagged (33.3%)`: the `social` ticket is scored with no message. `tiny_missing` ends with `KeyError: "['prior_escalations_90d'] not in index"` and `tiny_text` with `ValueError: Cannot use median strategy with non-numeric data`, each with exit code 1: the end of a long traceback. `tiny_cents` says `3 flagged (100.0%)`, with no message and exit code 0.

Two quiet cases: the new channel, and the order values in cents. **Input checks** stop such batches before scoring. **Monitoring** watches the results over time, for slow changes (**drift**).

## 2. Input checks in predict.py

The next cell writes the finished `predict.py`, the same as the course's [C05/project/predict.py](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/project/predict.py). Compared with the first version, it has `RANGES`, `check_input()` and seven new lines in `main()`. The [lesson page](https://learning.nextia-ai.com/courses/ml/m06/plan-production-checks/#input-checks-in-predictpy) explains each part.

In [ ]:
%%writefile predict.py
"""Score new tickets with the saved escalation model.

Nextia Learning, C05, Module 6 (Package artifacts; Plan production checks).

    python predict.py data/july_tickets.csv predictions.csv
    python predict.py --check     reload the model and compare the fixed sample

Checks the input before it scores: the required columns, numbers that are
numbers, values in their allowed range, and categories that the model knows.
A missing column or a wrong type stops the run (exit code 1). An unknown
category is allowed (the model treats it as "none of the known ones") but is
counted and reported, because many of them mean that the input has changed.
"""

import argparse
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from ticket_model import FEATURES, read

MODEL = Path("model")
RANGES = {"priority": (1, 3), "word_count": (0, 10000), "order_value": (0, 5000), "customer_tenure_days": (0, 20000),
          "prior_tickets_90d": (0, 500), "created_hour": (0, 23), "prior_escalations_90d": (0, 500)}


def load_model():
    info = json.loads((MODEL / "model_info.json").read_text(encoding="utf-8"))
    return joblib.load(MODEL / "escalation_model.joblib"), info


def check_input(df, info):
    """Return a list of problems that stop the run, and a list of warnings."""
    errors, warnings = [], []
    missing = [c for c in FEATURES if c not in df.columns]
    if missing:
        return [f"missing columns: {', '.join(missing)}"], warnings
    for name in info["features"]["numbers"]:
        values = pd.to_numeric(df[name], errors="coerce")
        bad = values.isna() & df[name].notna()
        if bad.any():
            errors.append(f"{name}: {bad.sum()} values are not numbers, for example {df.loc[bad, name].iloc[0]!r}")
            continue
        low, high = RANGES[name]
        outside = values.notna() & ((values < low) | (values > high))
        if outside.any():
            errors.append(f"{name}: {outside.sum()} values outside {low} to {high}")
    for name, known in info["features"]["categories"].items():
        unknown = df[name].notna() & ~df[name].isin(known)
        if unknown.any():
            values = ", ".join(sorted(df.loc[unknown, name].astype(str).unique())[:5])
            warnings.append(f"{name}: {unknown.sum()} rows have a category the model never saw ({values})")
    return errors, warnings


def check_sample(pipeline):
    sample = read(MODEL / "check_sample.csv")
    scores = pipeline.predict_proba(sample[FEATURES])[:, 1]
    largest = float(np.abs(scores - sample["score"]).max())
    if largest > 1e-9:
        sys.exit(f"Check failed: the reloaded model's scores differ from the saved ones by up to {largest:.2e}.")
    print(f"Check passed: {len(sample)} scores are identical after reloading (largest difference {largest:.1e}).")


def main():
    parser = argparse.ArgumentParser(description="Score new tickets with the saved escalation model.")
    parser.add_argument("input", nargs="?", help="CSV file with one row per new ticket")
    parser.add_argument("output", nargs="?", default="predictions.csv")
    parser.add_argument("--check", action="store_true", help="compare the fixed sample after reloading")
    args = parser.parse_args()
    pipeline, info = load_model()
    if args.check:
        check_sample(pipeline)
        return
    if not args.input:
        parser.error("give an input CSV file, or --check")

    df = read(args.input)
    errors, warnings = check_input(df, info)
    for warning in warnings:
        print(f"Warning: {warning}")
    if errors:
        for error in errors:
            print(f"Error: {error}")
        sys.exit("Stopped: the input does not match the model's feature schema. Nothing was scored.")

    scores = pipeline.predict_proba(df[FEATURES])[:, 1]
    out = pd.DataFrame({"ticket_id": df["ticket_id"], "score": scores.round(4),
                        "flag": (scores >= info["threshold"]).astype(int)})
    out.to_csv(args.output, index=False)
    print(f"Scored {len(out)} tickets with threshold {info['threshold']}: {out['flag'].sum()} flagged "
          f"({out['flag'].mean():.1%}). Wrote {args.output}.")


if __name__ == "__main__":
    main()

In [ ]:
expect_hash('predict.py', Path("predict.py").read_text(encoding="utf-8"), '0ae223f45f35e759')

> **Predict.** Which of the five files will the finished version stop? Which one gets a warning?

In [ ]:
for name in ("tiny_ok", "tiny_social", "tiny_missing", "tiny_text", "tiny_cents"):
    print(f"--- {name}.csv")
    run("predict.py", f"{name}.csv", "tiny_out.csv")

> **Check.** `tiny_social` gives `Warning: channel: 1 rows have a category the model never saw (social)` and is scored. `tiny_missing` gives `Error: missing columns: prior_escalations_90d`, `tiny_text` gives `Error: priority: 1 values are not numbers, for example 'high'`, and `tiny_cents` gives `Error: order_value: 3 values outside 0 to 5000`. Each of the three ends with `Stopped: ... Nothing was scored.` and exit code 1.

> **Your turn.** Make a broken copy of the July input: rename the column `priority` to `prio`, and save it as `broken.csv` (without the index). The check cell runs `predict.py` on it.

In [ ]:
broken = ...  # the July tickets with the column priority renamed to prio
# save it as broken.csv

In [ ]:
if not Path("broken.csv").exists():
    print("Not yet: broken.csv does not exist. Save it with broken.to_csv(...).")
else:
    result = subprocess.run([sys.executable, "predict.py", "broken.csv", "out.csv"], capture_output=True, text=True)
    print(result.stdout.strip())
    if result.returncode == 1 and "missing columns: priority" in result.stdout:
        print("Check passed: predict.py stops and names the missing column.")
    else:
        print("Not yet: predict.py did not stop on a missing priority column. Check the column name in broken.csv.")

## 3. Score the July tickets

> **Predict.** About 6.5% of the July tickets come from `social`. Will `predict.py` stop or warn?

In [ ]:
run("predict.py", "data/july_tickets.csv", "predictions.csv")

> **Check.** You should see `Warning: channel: 91 rows have a category the model never saw (social)` and `Scored 1400 tickets with threshold 0.19: 282 flagged (20.1%). Wrote predictions.csv.`

### Monitoring signals

A **monitoring signal** is a number that you compute for each period and compare with what you expect. These four need no labels: the flag rate (against the capacity of 20%), the mean score, the share of unknown categories, and the mean of a number feature. In May they were 19.3%, 0.120, 0 and 9.00 (for `prior_tickets_90d`).

In [ ]:
import json

info = json.load(open("model/model_info.json"))
scored = read("data/july_tickets.csv").merge(pd.read_csv("predictions.csv"), on="ticket_id")
scored["days"] = pd.cut(scored["created_at"].dt.day, [0, 7, 14, 21, 28, 31],
                        labels=["1-7", "8-14", "15-21", "22-28", "29-31"])
scored["unknown"] = ~scored["channel"].isin(info["features"]["categories"]["channel"])
signals = scored.groupby("days", observed=True).agg(
    tickets=("flag", "size"), flag_rate=("flag", "mean"), mean_score=("score", "mean"),
    unknown=("unknown", "mean"), prior_tickets=("prior_tickets_90d", "mean"))
print(signals.round(3))

> **Check.** For 1–7 and 8–14 July the flag rate is 0.174 and 0.185. For 15–21 July it is **0.262**, the mean score 0.138, and 12.6% of channels are unknown. `prior_tickets` falls from 5.322 to 3.033.

The next cell draws the flag rate over the last 7 days, for each day from 7 July, against the capacity.

In [ ]:
import matplotlib.pyplot as plt

daily = scored.groupby(scored["created_at"].dt.normalize()).agg(tickets=("flag", "size"), flagged=("flag", "sum"),
                                                                unknown=("unknown", "sum"))
last7 = daily.rolling(7).sum().dropna()
last7["flag_rate"] = last7["flagged"] / last7["tickets"]
last7["unknown_share"] = last7["unknown"] / last7["tickets"]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(last7.index.day, 100 * last7["flag_rate"], marker="o", label="Flag rate, last 7 days")
ax.axhline(20, linestyle="--", color="gray", label="Capacity 20%")
ax.set_xlabel("Day of July 2026"); ax.set_ylabel("Tickets flagged (%)"); ax.set_ylim(0, 30); ax.legend()
plt.show()

> **Check.** The line is under 20% until 14 July, crosses it on 15 July, and reaches 26.2% on 21 July.

> **Your turn.** Set `flag_rate_from_15` to the share of the tickets created from 15 July that are flagged, rounded to 3 decimals. (Use `scored`.)

In [ ]:
flag_rate_from_15 = ...  # the flag rate of the tickets created on or after 15 July, rounded to 3 decimals
print(flag_rate_from_15)

In [ ]:
expect_hash('flag_rate_from_15', flag_rate_from_15, '76924be2b83afe2d')

Is the new channel the only cause? The next cell compares the `social` tickets with the other tickets from 15 July.

In [ ]:
from_15 = scored[scored["created_at"].dt.day >= 15]
print("social:", round(from_15.loc[from_15["unknown"], "flag"].mean(), 3),
      "| other channels:", round(from_15.loc[~from_15["unknown"], "flag"].mean(), 3))

> **Check.** You should see `social: 0.231 | other channels: 0.218`. The other tickets are flagged more often too, so something else changed. The falling `prior_tickets_90d` is the clue: the July export counts only the customer's tickets before 1 July. The signals show *that* something changed and *where* to look; a person who knows the data finds the cause.

### When the labels arrive

The targets of the July tickets arrive later, in `july_labels.csv`. Now you can compute precision and recall.

In [ ]:
labelled = scored.merge(pd.read_csv("data/july_labels.csv"), on="ticket_id")
day = labelled["created_at"].dt.day
for name, part in (("1-14 July", day < 15), ("15-31 July", day >= 15), ("All July", day > 0)):
    d = labelled[part]
    caught = (d["flag"] & d["escalated_72h"]).sum()
    print(f"{name}: {len(d)} tickets, escalated {d['escalated_72h'].mean():.3f}, mean score {d['score'].mean():.3f},"
          f" precision {caught / d['flag'].sum():.3f}, recall {caught / d['escalated_72h'].sum():.3f}")

> **Check.** For all of July: escalated 0.140, mean score 0.125, precision **0.333**, recall **0.480**. From 15 July, precision falls to 0.312.

## 4. A written retraining trigger

The lesson's trigger starts a retraining review when one of these happens:

1. The flag rate of the last 7 days is above 20% on 5 working days in a row.
2. Unknown categories are more than 2% of the tickets of the last 7 days.
3. Recall is below 0.40 on a month with labels.

The next cell checks condition 1 for each working day (Monday to Friday).

In [ ]:
in_a_row, fired_1 = 0, None
for date, row in last7.iterrows():
    if date.weekday() >= 5:      # Saturday or Sunday
        continue
    in_a_row = in_a_row + 1 if row["flag_rate"] > 0.20 else 0
    if in_a_row == 5 and fired_1 is None:
        fired_1 = date.date()
print("condition 1 fires on:", fired_1)

> **Check.** You should see `condition 1 fires on: 2026-07-21`: the working days 15, 16, 17, 20 and 21 July.

> **Your turn.** Set `fired_2` to the first day (a `date`, as in the cell above) on which condition 2 is true: `last7["unknown_share"]` above 0.02.

In [ ]:
fired_2 = ...  # the first date with unknown_share above 0.02
print("condition 2 fires on:", fired_2)

In [ ]:
expect_hash('fired_2', str(fired_2), '43d138bb0adc5814')

Condition 3 does not fire: July recall is 0.480. The review finds two causes (the new channel and the export of `prior_tickets_90d`) and decides to retrain with May and June included. The [lesson page](https://learning.nextia-ai.com/courses/ml/m06/plan-production-checks/#a-written-retraining-trigger) has the full plan. How to run the checks on a schedule and roll back an artifact is deployment: C17, *Deploying AI Models for Real Users*.

## 5. Your turn: the review on 18 July

It is 18 July. The labels of 1 to 14 July have arrived. Use the outputs above. Which conditions have fired? What is the recall of 1 to 14 July? Should the review start? Write your answer in the next cell (double-click to edit), then compare with the lesson page.

*Your answer here:*

## 6. Failure case: order values in cents

An export change sends `order_value` in cents. The next cell scores such a file in this process, without checks, as the first version would.

> **Predict.** Will the flag rate go up or down? By a little or a lot?

In [ ]:
import joblib

pipeline = joblib.load("model/escalation_model.joblib")
july_cents = july.copy()
july_cents["order_value"] = july_cents["order_value"] * 100
july_cents.to_csv("july_cents.csv", index=False)
scores = pipeline.predict_proba(july_cents[FEATURES])[:, 1]
print("flag rate without checks:", round((scores >= 0.19).mean(), 3), "| mean score:", round(scores.mean(), 3))

> **Check.** You should see `flag rate without checks: 0.836 | mean score: 0.799`: more than four times the capacity, with no error. Now the finished `predict.py`:

In [ ]:
run("predict.py", "july_cents.csv", "out.csv")

> **Check.** You should see `Error: order_value: 879 values outside 0 to 5000` and `Stopped:`, with exit code 1. Small and missing orders stay in the range, so the check catches the unit error on most rows, not on every row.

## 7. Module practice: deliver the classifier

The task: deliver the classifier and an evaluation report, with commands that reproduce every number, and prove it in a **fresh copy**. The local route copies only the code and documents into a new folder, makes a new `.venv` and runs `get_data.py` there (the lesson page has the commands). Here, the next cell makes the fresh copy next to `ticket-model` and copies the code and the data (in place of `get_data.py`). The model and the reports are **not** copied: the commands make them again.

In [ ]:
import shutil

fresh = Path("..", "ticket-model-fresh").resolve()
shutil.rmtree(fresh, ignore_errors=True)
(fresh / "data").mkdir(parents=True)
for name in ("ticket_model.py", "train.py", "evaluate.py", "predict.py", "requirements.txt"):
    shutil.copy(name, fresh / name)
for csv in Path("data").glob("*.csv"):
    shutil.copy(csv, fresh / "data" / csv.name)
print(sorted(p.name for p in fresh.iterdir()))

> **Check.** You should see `['data', 'evaluate.py', 'predict.py', 'requirements.txt', 'ticket_model.py', 'train.py']`: the code and `data`, without `model` and `reports`.

Now run the commands in the fresh copy, each in a new process. You should see `Threshold 0.19`, `Check passed`, the test report (precision 0.357, recall 0.446) and `282 flagged (20.1%)`.

In [ ]:
for command in (["train.py"], ["predict.py", "--check"], ["evaluate.py"],
                ["predict.py", "data/july_tickets.csv", "predictions.csv"]):
    print("$ python", " ".join(command))
    run(*command, cwd=fresh)

> **Your turn.** The completion check: the same artifact must give identical predictions in a fresh process. Compute the SHA-256 of `predictions.csv` here and in the fresh copy (use `hashlib.sha256(path.read_bytes()).hexdigest()`), and set `same` to `True` if they are equal.

In [ ]:
import hashlib

here = ...   # SHA-256 of predictions.csv in this folder
there = ...  # SHA-256 of predictions.csv in the fresh copy
same = ...
print(here, there, same, sep="\n")

In [ ]:
expect_hash('same', same, 'b5bea41b6c623f7c')

> **Check.** On macOS with the course's versions, both checksums are `91936b4eff2c28a1b0c3382b8b88a05f2f7e677fc8395b2ab4486830a3dd9ff1`. On another system they can differ from this, but they must be equal to each other.

Write your evaluation report in the next cell (double-click to edit). Use the outputs of this notebook and of the [model card](https://learning.nextia-ai.com/courses/ml/m06/document-use-boundaries/). The lesson page has a model answer.

*Your evaluation report:*

| At threshold 0.19 | Validation | Test (once) | Rule "priority 1" (test) |
|---|---|---|---|
| Flagged | | | |
| Precision | | | |
| Recall | | | |

*Commands that reproduce it, with their expected output:*

## Check your understanding and recap

Answer the **knowledge checks** and the **module check** on the [lesson page](https://learning.nextia-ai.com/courses/ml/m06/plan-production-checks/#module-check). They count toward your certificate when you are signed in and enrolled.

`predict.py` now stops on a missing column, a wrong type or a value out of range, and warns about unknown categories. In July, the 7-day flag rate rose above the capacity from 15 July (26.2% on 21 July), unknown channels reached 12.6%, and `prior_tickets_90d` drifted. Two conditions of the retraining trigger fired. When the labels arrived, July precision was 0.333 and recall 0.480. The fresh copy gave identical predictions. Next: the [final assignment](https://learning.nextia-ai.com/courses/ml/final-assignment/).

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Final assignment](https://learning.nextia-ai.com/courses/ml/final-assignment/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m06/plan-production-checks/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.